In [1]:
import os
import re
import json
import glob
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

INPUT = "/kaggle/input/datasets/nocturnalnerd18"
img_dir = "/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_CACHE_DIR = "/kaggle/temp/vg"
VG_URLS = ["https://cs.stanford.edu/people/rak248/VG_100K", "https://cs.stanford.edu/people/rak248/VG_100K_2"]
os.makedirs(VG_CACHE_DIR, exist_ok=True)
OUT_DIR = "/kaggle/working/label_audit"
os.makedirs(OUT_DIR, exist_ok=True)
COLOURS = {"black", "white", "yellow", "red", "blue", "green", "brown", "gray", "grey", "orange", "pink", "purple", "silver", "gold"}
N_COLOUR, N_OTHER, PER_PAGE = 60, 40, 12
rng = np.random.default_rng(0)


VG_INDEX = {}
for root_ in glob.glob("/kaggle/input/**/*isual*enome*", recursive=True):
    if os.path.isdir(root_):
        for dp, dn, fn in os.walk(root_):
            for name in fn:
                if name.endswith(".jpg"):
                    VG_INDEX.setdefault(name, os.path.join(dp, name))
print(len(VG_INDEX), "Visual Genome images indexed from the attached dataset")


def find_vg_image_path(image_id):
    name = f"{image_id}.jpg"
    if name in VG_INDEX:
        return VG_INDEX[name]
    local = os.path.join(VG_CACHE_DIR, name)
    if os.path.exists(local):
        return local
    for attempt in range(3):
        for base in VG_URLS:
            try:
                r = requests.get(f"{base}/{image_id}.jpg", timeout=30)
            except requests.RequestException:
                continue
            if r.status_code == 200 and len(r.content) > 1000:
                with open(local + ".part", "wb") as f:
                    f.write(r.content)
                os.replace(local + ".part", local)
                return local
        time.sleep(2 * (attempt + 1))
    return None


def resolve_image_path(q):
    qid = q["question_id"]
    if qid.startswith("pope_"):
        return os.path.join(img_dir, f"{q['image_id']}.jpg")
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    if qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    question_by_id = {q["question_id"]: q for q in json.load(f)}
with open(f"{INPUT}/atp-llava-selection/selection.json") as f:
    selection = json.load(f)

pool = [q for q in selection["selected"]["attribute"] if selection["stratum"][q] == "false_yes"]
word = {q: (re.sub(r"[^a-z\- ]", "", re.sub(r"\s+in this image\??\s*$", "", question_by_id[q]["question_text"].lower())).split() or [""])[-1] for q in pool}
colour = [q for q in pool if word[q] in COLOURS]
other = [q for q in pool if word[q] not in COLOURS]
chosen = list(rng.choice(colour, size=min(N_COLOUR, len(colour)), replace=False)) + list(rng.choice(other, size=min(N_OTHER, len(other)), replace=False))
chosen = list(rng.permutation(chosen))
print(len(pool), "attribute false yes,", len(colour), "colour,", len(other), "other;", len(chosen), "chosen")

vg = sorted({question_by_id[q]["image_id"] for q in chosen if q.startswith("reefknot_")})
with ThreadPoolExecutor(max_workers=16) as ex:
    got = list(ex.map(find_vg_image_path, vg))
print("visual genome images fetched", len(vg) - sum(g is None for g in got), "of", len(vg))

108249 Visual Genome images indexed from the attached dataset
1736 attribute false yes, 336 colour, 1400 other; 100 chosen
visual genome images fetched 0 of 0


In [2]:
rows = []
n_pages = int(np.ceil(len(chosen) / PER_PAGE))
for page in range(n_pages):
    ids = chosen[page * PER_PAGE:(page + 1) * PER_PAGE]
    fig, axes = plt.subplots(3, 4, figsize=(20, 15.5))
    for ax, qid in zip(axes.ravel(), ids):
        q = question_by_id[qid]
        path = resolve_image_path(q)
        im = Image.open(path).convert("RGB") if path and os.path.exists(path) else None
        if im is not None:
            im.thumbnail((640, 640))
            ax.imshow(im)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"{qid}\n{q['question_text']}  [label: {q['ground_truth_answer']}]", fontsize=9)
        rows.append({"question_id": qid, "page": page, "word": word[qid], "question_text": q["question_text"], "ground_truth_answer": q["ground_truth_answer"], "image_found": im is not None})
    for ax in axes.ravel()[len(ids):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/audit_page_{page + 1}.jpg", dpi=70, pil_kwargs={"quality": 80})
    plt.close(fig)
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/audit_index.csv", index=False)
print(n_pages, "pages written to", OUT_DIR, "; image missing for", sum(not r["image_found"] for r in rows))

9 pages written to /kaggle/working/label_audit ; image missing for 0
